# Amazon FSTSP Instance Preparation
Prepares FSTSP instances from the Amazon Last-Mile Routing dataset.

**Design — proposal Section 4 (equations 12-15):**
- Only `route_data.json` and `eval_route_data.json` needed (no large travel_times.json)
- **Truck times**: Manhattan distance / $v_{\text{truck}}$ + noise (road-network proxy)
- **Drone times**: Euclidean distance / $v_{\text{drone}}$ + noise (straight-line flight)
- Times in **minutes**, `SCALE=10` (matches `Dis_Scale` in `Penalty_TSPMD.c`)
- Real `(hour, dow)` from `departure_time_utc` as context $S$

**Three splits:**
- **Train** (~6112): from `route_data.json` → train $g_\theta$
- **Val**   (~1526): from `eval_route_data.json` (50%) → tune hyperparameters
- **Test**  (~1526): from `eval_route_data.json` (50%) → report final performance

**Output:**
```
amazon_instances/
├── configs_train/   ← .tspmd + .par per training route
├── configs_val/     ← .tspmd + .par per val route
├── configs_test/    ← .tspmd + .par per test route
└── dataset/
    ├── data.npz           ← S_hat + W_hat for all three splits
    ├── oracle_val.npz     ← z_hat_oracle + z_oracle for val  (after Step 8)
    ├── oracle_test.npz    ← z_hat_oracle + z_oracle for test (after Step 8)
    ├── meta_train.csv
    ├── meta_val.csv
    ├── meta_test.csv
    └── W_hat_sample.csv
```

## Step 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, math, json, datetime, subprocess
import numpy as np, pandas as pd
from pathlib import Path

PROJECT      = "/content/drive/MyDrive/CSCI 619 project"
AMAZON_DIR   = f"{PROJECT}/amazon_instances"
CONFIG_TRAIN = f"{AMAZON_DIR}/configs_train"
CONFIG_VAL   = f"{AMAZON_DIR}/configs_val"
CONFIG_TEST  = f"{AMAZON_DIR}/configs_test"
DATASET_DIR  = f"{AMAZON_DIR}/dataset"
# CARC_TRAIN_DIR  = f"{AMAZON_DIR}/carc_par_train" #
# CARC_VAL_DIR  = f"{AMAZON_DIR}/carc_par_val"  #
# CARC_TEST_DIR = f"{AMAZON_DIR}/carc_par_test" #

# for d in [CONFIG_TRAIN, CONFIG_VAL, CONFIG_TEST, CARC_TRAIN_DIR,
#           CARC_VAL_DIR, CARC_TEST_DIR, DATASET_DIR]:
#     os.makedirs(d, exist_ok=True)
for d in [CONFIG_TRAIN, CONFIG_VAL, CONFIG_TEST, DATASET_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print("Output:", AMAZON_DIR)

Mounted at /content/drive
Drive mounted.
Output: /content/drive/MyDrive/CSCI 619 project/amazon_instances


In [ ]:
import subprocess, shutil, os

LKH_SRC_DIR = f"{PROJECT}/LKH-3.0.14"

# Delete old binary
if os.path.exists(f"{LKH_SRC_DIR}/LKH"):
    os.remove(f"{LKH_SRC_DIR}/LKH")

# Recompile
r = subprocess.run(["make", "-j2"], cwd=LKH_SRC_DIR,
                   capture_output=True, text=True)
print("OK" if r.returncode == 0 else r.stderr[-300:])

# Verify
r2 = subprocess.run(
    ["grep", "-c", "DcostPen",
     f"{LKH_SRC_DIR}/SRC/Penalty_TSPMD.c"],
    capture_output=True, text=True)
print(f"Linear cost lines found: {r2.stdout.strip()}  (should be 3)")

OK
Linear cost lines found: 3  (should be 3)


## Step 2 — Load data files

Only GPS coordinate files needed — no large travel_times.json required.

In [ ]:
# Option A: uploaded to /content/
# TRAIN_ROUTE_JSON = "/content/route_data.json"
# EVAL_ROUTE_JSON  = "/content/eval_route_data.json"

# Option B: already in Drive
TRAIN_ROUTE_JSON = f"{PROJECT}/route_data.json"
EVAL_ROUTE_JSON  = f"{PROJECT}/eval_route_data.json"

with open(TRAIN_ROUTE_JSON) as f: train_routes = json.load(f)
with open(EVAL_ROUTE_JSON)  as f: eval_routes  = json.load(f)

print(f"Train routes: {len(train_routes)}")
print(f"Eval  routes: {len(eval_routes)}")

rid   = list(train_routes.keys())[0]
rd    = train_routes[rid]
stops = rd['stops']
print(f"\nExample route: {rid[:50]}")
print(f"  departure: {rd['departure_time_utc']}, date: {rd['date_YYYY_MM_DD']}")
print(f"  depot: {sum(1 for s in stops if stops[s]['type']=='Station')}, "
      f"customers: {sum(1 for s in stops if stops[s]['type']=='Dropoff')}")

Train routes: 6112
Eval  routes: 3052

Example route: RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77
  departure: 16:02:10, date: 2018-07-27
  depot: 1, customers: 118


## Step 3 — Configuration

In [ ]:
# ── FSTSP structure (must match FSTSP_DFL_Training.ipynb) ────────────
n        = 9      # customers per instance
k_aug    = 10     # augmentation factor
n_orig   = n + 2  # depot(0), cust(1..9), dummy(10)
SCALE    = 10    # must match Dis_Scale in Penalty_TSPMD.c
SEED     = 42

# ── Travel time parameters (proposal eq. 12) ─────────────────────────
V_TRUCK  = 40.0   # km/h — truck speed
V_DRONE  = 80.0   # km/h — drone speed

# ── Noise parameters (proposal Section 4, equations 13-15) ───────────
SIGMA_TRUCK  = 0.20;  SIGMA_DRONE  = 0.05
BETA_TRUCK   = 0.12;  BETA_DRONE   = 0.04
GAMMA        = 0.05   # Mon/Fri peak

# ── Val/test split ratio ──────────────────────────────────────────────
VAL_RATIO    = 0.5    # 50% of eval routes → val, 50% → test

print(f"n={n}, k={k_aug}, n_orig={n_orig}, SCALE={SCALE}")
print(f"v_truck={V_TRUCK} km/h (Manhattan), v_drone={V_DRONE} km/h (Euclidean)")
print(f"Times in MINUTES × SCALE={SCALE} → integers for LKH-3")
print(f"σ_truck={SIGMA_TRUCK}, β_truck={BETA_TRUCK}")
print(f"σ_drone={SIGMA_DRONE}, β_drone={BETA_DRONE}, γ={GAMMA}")

# Split eval routes
eval_ids  = list(eval_routes.keys())
rng_split = np.random.default_rng(SEED)
shuffled  = rng_split.permutation(len(eval_ids))
n_val     = int(len(eval_ids) * VAL_RATIO)
val_ids   = [eval_ids[i] for i in shuffled[:n_val]]
test_ids  = [eval_ids[i] for i in shuffled[n_val:]]
val_routes  = {rid: eval_routes[rid] for rid in val_ids}
test_routes = {rid: eval_routes[rid] for rid in test_ids}

print(f"\nSplit: {len(train_routes)} train | "
      f"{len(val_routes)} val | {len(test_routes)} test")

n=9, k=10, n_orig=11, SCALE=10
v_truck=40.0 km/h (Manhattan), v_drone=80.0 km/h (Euclidean)
Times in MINUTES × SCALE=10 → integers for LKH-3
σ_truck=0.2, β_truck=0.12
σ_drone=0.05, β_drone=0.04, γ=0.05

Split: 6112 train | 1526 val | 1526 test


## Step 4 — Core functions

In [ ]:
def sample_customers(route_data_dict, route_id, rng, n_customers=9):
    """Sample n customers + depot from a route. Returns node_ids list."""
    stops     = route_data_dict[route_id]['stops']
    depot_ids = [s for s in stops if stops[s]['type'] == 'Station']
    cust_ids  = [s for s in stops if stops[s]['type'] == 'Dropoff']
    depot_id  = depot_ids[0] if depot_ids else list(stops.keys())[0]
    sampled   = list(rng.choice(cust_ids,
                                size=min(n_customers, len(cust_ids)),
                                replace=False))
    return [depot_id] + sampled   # length = n+1


def get_travel_times(route_data_dict, route_id, node_ids):
    """
    Baseline travel times τ̄_ij = d_ij / v  (proposal equation 12).
    Truck : Manhattan distance (road-network proxy) in minutes
    Drone : Euclidean distance (straight-line flight) in minutes
    """
    stops  = route_data_dict[route_id]['stops']
    ids    = node_ids + [node_ids[0]]   # dummy depot = depot
    lat_km = 111.0

    tau_bar_truck = np.zeros((n_orig, n_orig))
    tau_bar_drone = np.zeros((n_orig, n_orig))

    for i in range(n_orig):
        for j in range(n_orig):
            if i == j: continue
            a, b   = ids[i], ids[j]
            la, lo = stops[a]['lat'], stops[a]['lng']
            lb, ob = stops[b]['lat'], stops[b]['lng']
            lng_km = lat_km * math.cos(math.radians((la+lb)/2))

            dy = abs(la - lb) * lat_km            # N-S (km)
            dx = abs(lo - ob) * lng_km            # E-W (km)

            d_manhattan  = dx + dy                # truck: road proxy
            d_euclidean  = math.sqrt(dx**2+dy**2) # drone: direct

            tau_bar_truck[i,j] = (d_manhattan / V_TRUCK) * 60  # minutes
            tau_bar_drone[i,j] = (d_euclidean / V_DRONE) * 60  # minutes

    return tau_bar_truck, tau_bar_drone


def get_context_and_noise(route_data_dict, route_id, node_ids, rng):
    """
    Compute s_hat_i, tau_truck, tau_drone for one instance.
    Equations 12-15 from proposal. Times in minutes.
    """
    rd    = route_data_dict[route_id]
    stops = rd['stops']
    ids   = node_ids + [node_ids[0]]

    # Real (hour, dow) from route metadata
    dep  = rd['departure_time_utc']
    date = rd['date_YYYY_MM_DD']
    hour = float(dep.split(':')[0]) + float(dep.split(':')[1]) / 60.0
    dow  = datetime.date.fromisoformat(date).weekday()  # 0=Mon, 6=Sun

    # h(hour, dow) — equation (15)
    h_val  = max(0.0, math.sin(math.pi*(hour-6)/12.0))
    h_val += GAMMA * float(dow in [0, 4])

    # μ_k(S) — equation (14)
    mu_truck = BETA_TRUCK * h_val
    mu_drone = BETA_DRONE * h_val

    # Baseline τ̄_ij — equation (12), in minutes
    tau_bar_truck, tau_bar_drone = get_travel_times(
        route_data_dict, route_id, node_ids)

    # Realized τ_ij = τ̄_ij · exp(μ(S) + σ·ξ) — equation (13)
    xi_truck  = rng.standard_normal((n_orig, n_orig))
    xi_drone  = rng.standard_normal((n_orig, n_orig))
    tau_truck = tau_bar_truck * np.exp(mu_truck + SIGMA_TRUCK * xi_truck)
    tau_drone = tau_bar_drone * np.exp(mu_drone + SIGMA_DRONE * xi_drone)
    np.fill_diagonal(tau_truck, 0.0)
    np.fill_diagonal(tau_drone, 0.0)

    # Context features ŝ_i
    lats  = np.array([stops[ids[i]]['lat'] for i in range(n_orig)])
    lngs  = np.array([stops[ids[i]]['lng'] for i in range(n_orig)])
    lat_n = (lats - lats.mean()) / (lats.std() + 1e-8)
    lng_n = (lngs - lngs.mean()) / (lngs.std() + 1e-8)
    time_f = np.array([
        math.sin(math.pi * hour / 12),
        math.cos(math.pi * hour / 12),
        math.sin(2 * math.pi * dow / 7),
        math.cos(2 * math.pi * dow / 7),
    ])
    s_hat = np.concatenate([lat_n, lng_n, time_f]).astype(np.float32)

    return s_hat, tau_truck, tau_drone, hour, dow


def build_knn_structure(route_data_dict, route_id, node_ids, k=k_aug):
    """Replicate instance_generator.c k-NN logic in Python."""
    stops  = route_data_dict[route_id]['stops']
    ids    = node_ids + [node_ids[0]]   # dummy depot = depot
    lat_km = 111.0

    def dist(i, j):
        la, lo = stops[ids[i]]['lat'], stops[ids[i]]['lng']
        lb, ob = stops[ids[j]]['lat'], stops[ids[j]]['lng']
        lng_km = lat_km * math.cos(math.radians((la+lb)/2))
        return math.sqrt(((la-lb)*lat_km)**2 + ((lo-ob)*lng_km)**2)

    groups = {}; fake_to_orig = {1: 0}; fake_idx = 2
    for g in range(1, n+1): # range(1, 10) = [1, 2, ..., 9]
        # range(n_orig-1) = [0..9]: excludes dummy depot (index 10)
        # so DRAFT_LIMIT values are always 0-9, never 10
        candidates  = [i for i in range(n_orig - 1) if i != g]
        dists       = sorted([(dist(c, g), c) for c in candidates])
        group_nodes = []
        for _, orig in dists[:k-1]:
            fake_to_orig[fake_idx] = orig
            group_nodes.append(fake_idx)
            fake_idx += 1
        fake_to_orig[fake_idx] = g   # actual customer site (svc=0)
        group_nodes.append(fake_idx)
        fake_idx += 1
        groups[g] = group_nodes
    return groups, fake_to_orig

def write_tspmd(config_name, config_dir,
                groups, fake_to_orig, tau_truck, tau_drone):
    """Write .tspmd + Colab .par to config_dir, CARC .par to carc_dir."""
    N = 1 + n * k_aug
    W   = np.zeros((N, N), dtype=np.int64)
    svc = np.zeros(N,      dtype=np.int64)

    for a in range(1, N+1):
        oa = fake_to_orig[a]
        for b in range(1, N+1):
            W[a-1, b-1] = round(SCALE * tau_truck[oa, fake_to_orig[b]])
    for g, nodes in groups.items():
        for nd in nodes:
            svc[nd-1] = round(SCALE * tau_drone[fake_to_orig[nd], g])
        svc[nodes[-1]-1] = 0

    tspmd_path   = Path(config_dir) / f"{config_name}.tspmd"
    par_path     = Path(config_dir) / f"{config_name}.par"
    outtour_path = Path(config_dir) / f"{config_name}.outtour"

    # ── .tspmd ───────────────────────────────────────────────────────
    with open(tspmd_path, 'w') as f:
        f.write(f"NAME : {config_name}\nCOMMENT : Amazon FSTSP\n")
        f.write(f"TYPE : TSPMD\nDIMENSION : {N}\n")
        f.write("EDGE_WEIGHT_TYPE : EXPLICIT\nEDGE_WEIGHT_FORMAT : FULL_MATRIX\n")
        f.write(f"DRONES : 1\nEDGE_WEIGHT_SECTION\n")
        for row in W: f.write(" ".join(map(str, row)) + "\n")
        f.write("CTSP_SET_SECTION\n")
        for g, nodes in groups.items():
            f.write(f"{g} " + " ".join(map(str, nodes)) + " -1\n")
        f.write("SERVICE_TIME_SECTION\n")
        for i in range(N): f.write(f"{i+1} {svc[i]}\n")
        f.write("DRAFT_LIMIT_SECTION\n")
        for i in range(1, N+1): f.write(f"{i} {fake_to_orig[i]}\n")
        f.write("DEPOT_SECTION\n1\n-1\nEOF\n")

print("All functions defined.")
print(f"  Truck: Manhattan distance / {V_TRUCK} km/h × 60 × SCALE={SCALE}")
print(f"  Drone: Euclidean distance / {V_DRONE} km/h × 60 × SCALE={SCALE}")

All functions defined.
  Truck: Manhattan distance / 40.0 km/h × 60 × SCALE=10
  Drone: Euclidean distance / 80.0 km/h × 60 × SCALE=10


## Step 5 — Generate instances (train / val / test)

1 instance per route. Truck = Manhattan + noise. Drone = Euclidean + noise.

In [ ]:
def process_routes(route_data_dict, config_dir,
                   split_name, save_tspmd=True):
    """Process all routes → (S_hat, W_hat, metadata). 1 instance per route."""
    rng = np.random.default_rng(SEED)
    S_hat_list, W_hat_list, meta_list = [], [], []
    skipped = 0

    for idx, (route_id, rd) in enumerate(route_data_dict.items()):
        stops    = rd['stops']
        cust_ids = [s for s in stops if stops[s]['type'] == 'Dropoff']
        if len(cust_ids) < n:
            skipped += 1; continue

        node_ids = sample_customers(route_data_dict, route_id, rng, n)
        s_hat, tau_truck, tau_drone, hour, dow = get_context_and_noise(
            route_data_dict, route_id, node_ids, rng)
        w_hat = np.concatenate([tau_truck.flatten(),
                                 tau_drone.flatten()]).astype(np.float32)

        config_name = f"{split_name}_{idx}"

        if save_tspmd:
            groups, fake_to_orig = build_knn_structure(
                route_data_dict, route_id, node_ids)
            write_tspmd(config_name, config_dir,
                        groups, fake_to_orig, tau_truck, tau_drone)

        S_hat_list.append(s_hat)
        W_hat_list.append(w_hat)
        meta_list.append({
            'config_name': config_name,
            'route_id':    route_id,
            'hour':        round(hour, 2),
            'dow':         dow,
            'depot':       node_ids[0],
            'customers':   ','.join(node_ids[1:]),
        })

        if (idx+1) % 500 == 0:
            print(f"  {split_name}: {idx+1}/{len(route_data_dict)}")

    print(f"  {split_name}: {len(S_hat_list)} instances  ({skipped} skipped)")
    return (np.array(S_hat_list), np.array(W_hat_list),
            pd.DataFrame(meta_list))

print("Processing training routes (~6112)...")
S_hat_train, W_hat_train, meta_train = process_routes(
    train_routes, CONFIG_TRAIN, "train", save_tspmd=True)

print("\nProcessing val routes (~1526, hyperparameter tuning)...")
S_hat_val, W_hat_val, meta_val = process_routes(
    val_routes, CONFIG_VAL, "val", save_tspmd=True)

print("\nProcessing test routes (~1526, final performance report)...")
S_hat_test, W_hat_test, meta_test = process_routes(
    test_routes, CONFIG_TEST, "test", save_tspmd=True)

feat_dim = S_hat_train.shape[1]
cost_dim = W_hat_train.shape[1]
print(f"\ndim(ŝ_i) = {feat_dim}  [lat×{n_orig} + lng×{n_orig} + 4 time]")
print(f"dim(ŵ_i) = {cost_dim} [τ_truck({n_orig}²) + τ_drone({n_orig}²)]")

Processing training routes (~6112)...
  train: 500/6112
  train: 1000/6112
  train: 1500/6112
  train: 2000/6112
  train: 2500/6112
  train: 3000/6112
  train: 3500/6112
  train: 4000/6112
  train: 4500/6112
  train: 5000/6112
  train: 5500/6112
  train: 6000/6112
  train: 6112 instances  (0 skipped)

Processing val routes (~1526, hyperparameter tuning)...
  val: 500/1526
  val: 1000/1526
  val: 1500/1526
  val: 1526 instances  (0 skipped)

Processing test routes (~1526, final performance report)...
  test: 500/1526
  test: 1000/1526
  test: 1500/1526
  test: 1526 instances  (0 skipped)

dim(ŝ_i) = 26  [lat×11 + lng×11 + 4 time]
dim(ŵ_i) = 242 [τ_truck(11²) + τ_drone(11²)]


## Step 6 — Save to Drive

To load in `FSTSP_DFL_Training.ipynb`:
```python
data        = np.load(f"{AMAZON_DIR}/dataset/data.npz")
S_hat_train = data['S_hat_train'];  W_hat_train = data['W_hat_train']
S_hat_val   = data['S_hat_val'];    W_hat_val   = data['W_hat_val']
S_hat_test  = data['S_hat_test'];   W_hat_test  = data['W_hat_test']
```

In [ ]:
np.savez(f"{DATASET_DIR}/data.npz",
         S_hat_train=S_hat_train, W_hat_train=W_hat_train,
         S_hat_val=S_hat_val,     W_hat_val=W_hat_val,
         S_hat_test=S_hat_test,   W_hat_test=W_hat_test)

meta_train.to_csv(f"{DATASET_DIR}/meta_train.csv", index=False)
meta_val.to_csv(  f"{DATASET_DIR}/meta_val.csv",   index=False)
meta_test.to_csv( f"{DATASET_DIR}/meta_test.csv",  index=False)

# Readable sample for Google Sheets
s_cols  = ([f"lat_{i}" for i in range(n_orig)] +
           [f"lng_{i}" for i in range(n_orig)] +
           ["sin_hour","cos_hour","sin_dow","cos_dow"])
tt_cols = [f"tau_truck_{i}_{j}" for i in range(n_orig) for j in range(n_orig)]
td_cols = [f"tau_drone_{i}_{j}" for i in range(n_orig) for j in range(n_orig)]
w_cols  = tt_cols + td_cols
pd.DataFrame(W_hat_train[:10], columns=w_cols).to_csv(
    f"{DATASET_DIR}/W_hat_sample.csv", index=False)

print(f"Saved to {DATASET_DIR}/")
print(f"  Train: {len(S_hat_train)} instances  → configs_train/")
print(f"  Val:   {len(S_hat_val)}  instances  → configs_val/   (tune λ, σ)")
print(f"  Test:  {len(S_hat_test)} instances  → configs_test/  (final results)")

Saved to /content/drive/MyDrive/CSCI 619 project/amazon_instances/dataset/
  Train: 6112 instances  → configs_train/
  Val:   1526  instances  → configs_val/   (tune λ, σ)
  Test:  1526 instances  → configs_test/  (final results)


## Step 7 — Sanity check

In [ ]:
i     = 0
row   = meta_train.iloc[i]
w     = W_hat_train[i]
tau_t = w[:n_orig**2].reshape(n_orig, n_orig)
tau_d = w[n_orig**2:].reshape(n_orig, n_orig)
config = row['config_name']

print("Instance 0 (train):")
print("  Route:    ", row['route_id'][:50])
print("  Departure: hour={:.1f}, dow={} (0=Mon)".format(row['hour'], row['dow']))
print("  Config:   ", config)
print()
print("  .tspmd:", os.path.exists(CONFIG_TRAIN + "/" + config + ".tspmd"))
print("  .par:  ", os.path.exists(CONFIG_TRAIN + "/" + config + ".par"))
print()
print("tau_truck (minutes) — depot row:")
for j in range(1, n+1):
    print("  depot->cust{}: {:.2f} min  (in .tspmd: {})".format(
          j, tau_t[0,j], round(SCALE*tau_t[0,j])))
print()
print("tau_drone (minutes) — depot row:")
for j in range(1, n+1):
    print("  depot->cust{}: {:.2f} min".format(j, tau_d[0,j]))
print()
print("tau_drone symmetric:", np.allclose(tau_d, tau_d.T, atol=1e-4))


Instance 0 (train):
  Route:     RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77
  Departure: hour=16.0, dow=4 (0=Mon)
  Config:    train_0

  .tspmd: True
  .par:   False

tau_truck (minutes) — depot row:
  depot->cust1: 43.80 min  (in .tspmd: 438)
  depot->cust2: 41.59 min  (in .tspmd: 416)
  depot->cust3: 36.01 min  (in .tspmd: 360)
  depot->cust4: 44.25 min  (in .tspmd: 443)
  depot->cust5: 42.15 min  (in .tspmd: 422)
  depot->cust6: 30.97 min  (in .tspmd: 310)
  depot->cust7: 40.03 min  (in .tspmd: 400)
  depot->cust8: 30.95 min  (in .tspmd: 310)
  depot->cust9: 43.31 min  (in .tspmd: 433)

tau_drone (minutes) — depot row:
  depot->cust1: 11.71 min
  depot->cust2: 12.58 min
  depot->cust3: 12.01 min
  depot->cust4: 11.20 min
  depot->cust5: 12.45 min
  depot->cust6: 12.73 min
  depot->cust7: 13.34 min
  depot->cust8: 14.12 min
  depot->cust9: 14.32 min

tau_drone symmetric: False


## Step 8 — Solve oracle for all instances (run once in CARC, Center for Advanced Research Computing)

Calls LKH-3 on each instance under **real** $\hat{w}_i$ to get:
- $\hat{z}_i^*$ = oracle effective solution (shape 242)
- $z_i^* = \hat{w}_i^\top \hat{z}_i^*$ = oracle cost (scalar, minutes)

**Val oracle** → use for hyperparameter tuning ($\lambda$, $\sigma$, etc.)
**Test oracle** → use for final regret reported in paper

## Step 9 — Save oracle solution $z^*$ for all instances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, json, pickle
import numpy as np, pandas as pd
import torch
from pathlib import Path

PROJECT      = "/content/drive/MyDrive/CSCI 619 project"
AMAZON_DIR   = f"{PROJECT}/amazon_instances"
CONFIG_TRAIN = f"{AMAZON_DIR}/configs_train"
CONFIG_VAL   = f"{AMAZON_DIR}/configs_val"
CONFIG_TEST  = f"{AMAZON_DIR}/configs_test"
DATASET_DIR  = f"{AMAZON_DIR}/dataset"

n      = 9
n_orig = n + 2

data        = np.load(f"{DATASET_DIR}/data.npz")
S_hat_train = data['S_hat_train'];  W_hat_train = data['W_hat_train']
S_hat_val   = data['S_hat_val'];    W_hat_val   = data['W_hat_val']
S_hat_test  = data['S_hat_test'];   W_hat_test  = data['W_hat_test']

meta_train = pd.read_csv(f"{DATASET_DIR}/meta_train.csv")
meta_val   = pd.read_csv(f"{DATASET_DIR}/meta_val.csv")
meta_test  = pd.read_csv(f"{DATASET_DIR}/meta_test.csv")

print(f"Train: {len(S_hat_train)}, Val: {len(S_hat_val)}, Test: {len(S_hat_test)}")

# Verify .outtour files are present
for split, d in [("train", CONFIG_TRAIN),
                 ("val",   CONFIG_VAL),
                 ("test",  CONFIG_TEST)]:
    n_out = len(list(Path(d).glob("*.outtour")))
    print(f"  {split}: {n_out} .outtour files")

Train: 6112, Val: 1526, Test: 1526
  train: 6112 .outtour files
  val: 1526 .outtour files
  test: 1526 .outtour files


In [ ]:
def load_structure(tspmd_path):
    node_to_group={1:0}; groups={}; fake_to_orig={}; svc_base={}
    in_ctsp=in_svc=in_draft=False; N_total=0
    with open(tspmd_path) as f:
        for line in f:
            if "DIMENSION" in line and ":" in line:
                try: N_total=int(line.split(":")[1])
                except: pass
            if "CTSP_SET_SECTION"     in line: in_ctsp=True;  in_svc=in_draft=False; continue
            if "SERVICE_TIME_SECTION" in line: in_svc=True;   in_ctsp=in_draft=False; continue
            if "DRAFT_LIMIT_SECTION"  in line: in_draft=True; in_svc=in_ctsp=False;  continue
            if "DEPOT_SECTION"        in line: break
            if in_ctsp:
                p=line.split()
                if p and p[-1]=="-1":
                    g=int(p[0]); nodes=[int(x) for x in p[1:-1]]
                    groups[g]=nodes
                    for nd in nodes: node_to_group[nd]=g
            if in_svc:
                p=line.split()
                if len(p)==2: svc_base[int(p[0])]=float(p[1])
            if in_draft:
                p=line.split()
                if len(p)==2: fake_to_orig[int(p[0])]=int(p[1])
    saved_colors={}
    for nd in range(1, N_total+1):
        g=node_to_group.get(nd,0)
        saved_colors[nd]=-g if (g>0 and svc_base.get(nd,1.0)==0.0) else g
    return dict(N_total=N_total, node_to_group=node_to_group,
                groups=groups, fake_to_orig=fake_to_orig,
                saved_colors=saved_colors)


def decode_outtour_to_z_hat(outtour_path, struct):
    tour_all=[]; in_tour=False
    with open(outtour_path) as f:
        for line in f:
            if "TOUR_SECTION" in line: in_tour=True; continue
            if in_tour:
                v=line.strip()
                if v in("-1","EOF"): break
                nd=int(v)
                if 1<=nd<=struct['N_total']: tour_all.append(nd)

    # Guard: some CARC jobs may have timed out before writing tour
    if not tour_all:
        raise ValueError(f"Empty TOUR_SECTION in {outtour_path}")

    color_count=[0]*(n+1); truck_route=[]; group_launch={}
    for nd in tour_all:
        truck_route.append(nd)
        g=struct['node_to_group'].get(nd,0)
        if g>0:
            sc=struct['saved_colors'].get(nd,g)
            if sc<0: color_count[g]=2
            else:
                color_count[g]+=1
                if color_count[g]==1: group_launch[g]=nd
        if all(color_count[g]>=2 for g in range(1,n+1)): break
    truck_route.append(truck_route[0])

    z_truck=np.zeros((n_orig,n_orig)); z_drone=np.zeros((n_orig,n_orig))
    for i in range(len(truck_route)-1):
        a,b=truck_route[i],truck_route[i+1]
        z_truck[struct['fake_to_orig'][a], struct['fake_to_orig'][b]]+=1
    c2=[0]*(n+1); gl={}
    for nd in truck_route[:-1]:
        g=struct['node_to_group'].get(nd,0)
        if g==0: continue
        sc=struct['saved_colors'].get(nd,g)
        if sc<0: c2[g]=2; continue
        c2[g]+=1
        if c2[g]==1: gl[g]=nd
        elif c2[g]==2:
            z_drone[struct['fake_to_orig'][gl[g]],g]+=1
            z_drone[struct['fake_to_orig'][nd],   g]+=1
    return np.concatenate([z_truck.flatten(),
                           z_drone.flatten()]).astype(np.float32)

print("Helper functions defined.")

Helper functions defined.


In [ ]:
def build_oracle(meta_df, W_hat, config_dir, oracle_path, split_name):
    if os.path.exists(oracle_path):
        print(f"{split_name} oracle already cached — skipping."); return

    print(f"Building {split_name} oracle ({len(meta_df)} instances)...")
    z_hat_list=[]; z_star_list=[]; missing=[]

    for idx, row in enumerate(meta_df.itertuples(index=False)):
        config_name  = row.config_name
        outtour_path = f"{config_dir}/{config_name}.outtour"
        tspmd_path   = f"{config_dir}/{config_name}.tspmd"

        if not os.path.exists(outtour_path):
            missing.append(config_name); continue

        struct = load_structure(tspmd_path)
        z_hat  = decode_outtour_to_z_hat(outtour_path, struct)
        z_star = float(np.dot(W_hat[idx].astype(np.float64), z_hat))

        z_hat_list.append(z_hat)
        z_star_list.append(z_star)

        if (idx+1) % 500 == 0:
            print(f"  {idx+1}/{len(meta_df)}  "
                  f"avg z* = {np.mean(z_star_list):.2f} min")

    if missing:
        print(f"  WARNING: {len(missing)} missing — first 5: {missing[:5]}")

    np.savez(oracle_path,
             z_hat_oracle=np.array(z_hat_list),
             z_oracle=np.array(z_star_list))
    print(f"Saved → {oracle_path}  "
          f"(avg z*={np.mean(z_star_list):.2f} min)")


build_oracle(meta_train, W_hat_train, CONFIG_TRAIN,
             f"{DATASET_DIR}/oracle_train.npz", "train")

build_oracle(meta_val, W_hat_val, CONFIG_VAL,
             f"{DATASET_DIR}/oracle_val.npz", "val")

build_oracle(meta_test, W_hat_test, CONFIG_TEST,
             f"{DATASET_DIR}/oracle_test.npz", "test")

Building train oracle (6112 instances)...
  500/6112  avg z* = 82.14 min
  1000/6112  avg z* = 82.17 min
  1500/6112  avg z* = 81.60 min
  2000/6112  avg z* = 82.17 min
  2500/6112  avg z* = 81.88 min
  3000/6112  avg z* = 82.01 min
  3500/6112  avg z* = 82.11 min
  4000/6112  avg z* = 82.35 min
  4500/6112  avg z* = 82.17 min
  5000/6112  avg z* = 82.33 min
  5500/6112  avg z* = 82.18 min
  6000/6112  avg z* = 81.93 min
Saved → /content/drive/MyDrive/CSCI 619 project/amazon_instances/dataset/oracle_train.npz  (avg z*=81.96 min)
Building val oracle (1526 instances)...
  500/1526  avg z* = 79.55 min
  1000/1526  avg z* = 80.42 min
  1500/1526  avg z* = 80.42 min
Saved → /content/drive/MyDrive/CSCI 619 project/amazon_instances/dataset/oracle_val.npz  (avg z*=80.54 min)
Building test oracle (1526 instances)...
  500/1526  avg z* = 80.40 min
  1000/1526  avg z* = 80.96 min
  1500/1526  avg z* = 80.99 min
Saved → /content/drive/MyDrive/CSCI 619 project/amazon_instances/dataset/oracle_test.n

In [ ]:
class PrecomputedOptDataset(torch.utils.data.Dataset):
    def __init__(self, S_hat, W_hat, z_hat_oracle, z_oracle):
        self.S_hat        = torch.FloatTensor(S_hat)
        self.W_hat        = torch.FloatTensor(W_hat)
        self.z_hat_oracle = torch.FloatTensor(z_hat_oracle)
        self.z_oracle     = torch.FloatTensor(z_oracle).view(-1,1)
    def __len__(self): return len(self.S_hat)
    def __getitem__(self, idx):
        return (self.S_hat[idx], self.W_hat[idx],
                self.z_hat_oracle[idx], self.z_oracle[idx])


oracle_train = np.load(f"{DATASET_DIR}/oracle_train.npz")
oracle_val   = np.load(f"{DATASET_DIR}/oracle_val.npz")
oracle_test  = np.load(f"{DATASET_DIR}/oracle_test.npz")

dataset_train = PrecomputedOptDataset(
    S_hat_train, W_hat_train,
    oracle_train['z_hat_oracle'], oracle_train['z_oracle'])

dataset_val = PrecomputedOptDataset(
    S_hat_val, W_hat_val,
    oracle_val['z_hat_oracle'], oracle_val['z_oracle'])

dataset_test = PrecomputedOptDataset(
    S_hat_test, W_hat_test,
    oracle_test['z_hat_oracle'], oracle_test['z_oracle'])

ORACLE_PKL = f"{DATASET_DIR}/oracle.pkl"
with open(ORACLE_PKL, 'wb') as f:
    pickle.dump((dataset_train, dataset_val, dataset_test), f)

print(f"Saved oracle.pkl → {ORACLE_PKL}")
print(f"  train: {len(dataset_train)}, val: {len(dataset_val)}, "
      f"test: {len(dataset_test)}")

# Quick sanity check
w0    = W_hat_train[0].astype(np.float64)
zh0   = oracle_train['z_hat_oracle'][0].astype(np.float64)
z0    = oracle_train['z_oracle'][0]
check = np.dot(w0, zh0)
print(f"\nConsistency: w_hat^T z_hat = {check:.4f}, "
      f"z_oracle = {z0:.4f}  "
      f"{'✓' if abs(check-z0)<0.01 else '✗ MISMATCH'}")

Saved oracle.pkl → /content/drive/MyDrive/CSCI 619 project/amazon_instances/dataset/oracle.pkl
  train: 6112, val: 1526, test: 1526

Consistency: w_hat^T z_hat = 80.8926, z_oracle = 80.8926  ✓
